# Repository edition: 09_precision

Run from this repository in your existing GPU environment. Source is in `../suites/09_precision/`. The launcher creates a versioned working copy. No embedded base64 code or installation is needed. Review configuration before launching; optional stages are not all enabled by default. See `../docs/EXPERIMENTS.md` for paper mapping.


In [ ]:
from pathlib import Path
import sys
_here = Path.cwd().resolve()
_candidates = [_here, *_here.parents]
SUPPLEMENTARY = next((p for p in _candidates if (p / "tools" / "notebook_sources.py").exists()), None)
if SUPPLEMENTARY is None:
    SUPPLEMENTARY = next((p / "supplementary" for p in _candidates if (p / "supplementary/tools/notebook_sources.py").exists()), None)
if SUPPLEMENTARY is None:
    raise FileNotFoundError("Open this notebook from inside the extracted supplementary repository.")
sys.path.insert(0, str(SUPPLEMENTARY / "tools"))
from notebook_sources import source_bundle as _source_bundle


# Certificates, recalibration, 8B freezes and anisotropy
Full code and your existing cohort/model pins/input tables are embedded. **No installation or token setup; no old 7 GB upload required.**

1. Certificates: 16 prompts per selected model, three depths, three epsilons, seven weight families.
2. Report-only calibration: preserve independent labels; add subject-held-out exploratory recalibration and an 8B/OLMo restricted view.
3. Full-cohort Llama8B singleton freezes.
4. Optional fresh 1B anisotropy captures (100 evaluation prompts; trained + three initialization seeds).

The old null summaries cannot support CPU-only anisotropy: they lack vectors. A signed residual-gradient cosine and a nonnegative weight-gradient projection are defined separately. Instruct pairs are omitted; retain family-level observational wording.

Read `README.txt` in the extracted code for definitions, tolerances and limitations. These sampled certificates validate numerical implementation; they do not prove exactness for every possible input.


In [ ]:
from pathlib import Path
import base64,hashlib,io,json,os,subprocess,sys,time,uuid,zipfile
BUNDLE_SHA256 = hashlib.sha256(base64.b64decode(_source_bundle('09_precision'))).hexdigest()
BUNDLE = _source_bundle('09_precision')

payload=base64.b64decode(BUNDLE)
assert hashlib.sha256(payload).hexdigest()==BUNDLE_SHA256
CODE=Path.home()/('threeways_precision_code_'+BUNDLE_SHA256[:12]);CODE.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    for item in z.infolist():
        if CODE.resolve() not in (CODE/item.filename).resolve().parents:raise ValueError('Unsafe archive path')
    z.extractall(CODE)
EVIDENCE=CODE/'evidence'
print('Code and evidence ready:',CODE)


## 1. Existing GPU environment

In [ ]:
candidates=[Path('/home/ubuntu/venvs/threeways/bin/python'),Path(sys.executable),Path.home()/'threeways_causal_env/bin/python']
check="import torch,transformers,numpy,pandas,matplotlib,huggingface_hub; assert transformers.__version__=='4.48.3', 'Expected tested Transformers 4.48.3'; assert torch.cuda.is_available(), 'CUDA unavailable'; print(torch.cuda.get_device_name())"
PYTHON=None;errors=[]
for candidate in dict.fromkeys(candidates):
    if not candidate.exists():continue
    p=subprocess.run([str(candidate),'-c',check],capture_output=True,text=True)
    if p.returncode==0:
        PYTHON=candidate;print('Using:',candidate,'|',p.stdout.strip());break
    errors.append(str(candidate)+': '+p.stderr[-1000:])
if PYTHON is None:raise RuntimeError('Point candidates to your existing environment.\n'+'\n'.join(errors))


## 2. Configure ONCE
Defaults run certificates for all six models, then 8B freezes, then anisotropy. Set `RUN_ANISOTROPY=False` to defer those new captures. Set `FREEZE_PROMPTS=100` for an explicit smaller 8B study; `None` uses all 2,995 prompts.

Certificates can be expensive: up to **6,048 live/frozen forward evaluations per model**, plus gradient/capture work. No runtime promise has been made. There is one random rank-one direction and one gradient-aligned direction for each nonzero matrix gradient. Numerical failures and unresolved tiny slopes are retained in the audit.

Recalibration uses five held-out subject folds, min5/min20 success thresholds and three quantiles. It is exploratory; more coverage does not imply reliable calibration. Profiles under new labels need raw per-prompt gradients; the code reports whether those were found.

The configuration cell refuses to create an accidental second campaign in this kernel. To deliberately start a new one later, first run `del RESULTS` after checking the old queue has exited.


In [ ]:
from datetime import datetime
import shutil
if 'RESULTS' in globals():
    raise RuntimeError(f'Campaign already selected: {RESULTS}. Use monitor/export; do not rerun configuration.')
CERTIFICATE_MODELS=['llama1b','llama3b','llama8b','qwen7b','qwen3b','olmo7b']
RUN_8B_FREEZE=True
FREEZE_PROMPTS=None
RUN_ANISOTROPY=True

# Optional raw-profile directories: used read-only to relabel per-prompt splits.
RAW_PROFILE_ROOTS=[]  # Label/coverage analysis needs no raw results.
# Optional, potentially substantial CPU work to rebuild gradient-by-regime summaries:
# RAW_PROFILE_ROOTS=[Path.home()/'threeways_channel_followups/reviewer_v1',
#     Path.home()/'threeways_overnight_results/overnight_20261104_094937_00cb7442']

RUN_NAME='precision_'+datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:8]
RESULTS=Path.home()/'threeways_precision_results'/RUN_NAME
RESULTS.mkdir(parents=True,exist_ok=False)
CONFIG=json.loads((CODE/'followup_study/config.json').read_text())
CONFIG.update(certificate_prompts=16,certificate_layer_fractions=[0,.5,1],
              fd_epsilons=[1e-5,1e-4,1e-3],seeds=[0],freeze_mode='singletons',
              freeze_models=['llama8b'],freeze_prompts=FREEZE_PROMPTS,
              anisotropy_prompts=100,anisotropy_reference_subjects=100,
              anisotropy_init_seeds=[1729,1730,1731])
known={m['key'] for m in CONFIG['models']}
if not set(CERTIFICATE_MODELS)<=known:raise ValueError('Unknown model key')
JOBS=[dict(name=f'{i+1:02d}_certificate_{model}_{RUN_NAME}',models=[model],stage='certificate')
      for i,model in enumerate(CERTIFICATE_MODELS)]
if RUN_8B_FREEZE:JOBS.append(dict(name=f'{len(JOBS)+1:02d}_freeze_llama8b_{RUN_NAME}',models=['llama8b'],stage='freeze'))
if RUN_ANISOTROPY:JOBS.append(dict(name=f'{len(JOBS)+1:02d}_anisotropy_llama1b_{RUN_NAME}',models=['llama1b'],stage='anisotropy'))
PLAN=dict(root=str(RESULTS),evidence=str(EVIDENCE),raw_profiles=[str(p) for p in RAW_PROFILE_ROOTS],
          config=CONFIG,jobs=JOBS,minimum_free_gib=20,bundle_sha256=BUNDLE_SHA256)
PLAN_PATH=RESULTS/'plan.json';PLAN_PATH.write_text(json.dumps(PLAN,indent=2))
print('Selected campaign:',RESULTS)
print('Free disk GiB:',round(shutil.disk_usage(RESULTS).free/2**30,1))
for job in JOBS:print(job['name'])
print('CPU calibration runs alongside the GPU queue; source tables and cohorts are bundled.')


## 3. Launch detached queue
Keep the GPU instance powered on. Closing the browser does not stop the queue. The free-space floor is checked before every GPU job; it is not a model-download size guarantee.


In [ ]:
if (RESULTS/'started.json').exists():raise RuntimeError('Already launched. Use monitor/export.')
if 'queue_process' in globals() and queue_process.poll() is None:raise RuntimeError('Queue is running. Use monitor.')
with (RESULTS/'queue.log').open('a') as log:
    queue_process=subprocess.Popen([str(PYTHON),'-u','-m','precision_study.queue','--plan',str(PLAN_PATH)],
        cwd=CODE,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
print('Queue PID:',queue_process.pid)
print('Results:',RESULTS)


## 4. Monitor the CURRENT job

In [ ]:
def tail(path,size=5000):
    if not path.exists():return
    print('\n---',path.name,'---')
    with path.open('rb') as f:
        f.seek(max(0,path.stat().st_size-size));print(f.read().decode(errors='replace'))
for job in PLAN['jobs']:
    folder=RESULTS/job['name'];exit_path=folder/'exit.json'
    exit_info=json.loads(exit_path.read_text()) if exit_path.exists() else None
    print(job['name'],'| completion:',(folder/'completion.json').exists(),'| exit:',exit_info)
tail(RESULTS/'queue.log')
active_path=RESULTS/'active.json'
if active_path.exists():
    active=json.loads(active_path.read_text());print('Most recently started:',active['job']);tail(Path(active['log']))
tail(RESULTS/'cpu.log',3000)
exit_path=RESULTS/'exit.json'
print('QUEUE EXIT:',json.loads(exit_path.read_text()) if exit_path.exists() else 'Not recorded yet')


## 5. Print completed audits and calibration coverage
`returncode=0` means execution succeeded, not that every numerical certificate passed. Look at the explicit failure and unresolved counts below. Sparse reference sets remain labelled uncalibrated.


In [ ]:
import csv
for job in PLAN['jobs']:
    root=RESULTS/job['name']
    for name in ['certificate_audit.json','checks.json']:
        p=root/'reports'/name
        if p.exists():print('\n',job['name'],name,json.loads(p.read_text()))
p=RESULTS/'00_recalibration/calibration_coverage_comparison.csv'
if p.exists():
    print('\nCalibration coverage:')
    with p.open() as f:
        for row in csv.DictReader(f):print(row)
p=RESULTS/'00_recalibration/status.json'
if p.exists():print('\nCalibration/report status:',json.loads(p.read_text()))


## 6. Stop only if wanted
Uncomment to request a cooperative stop. Wait for queue exit before shutting down. A finite-difference batch or reference capture may finish first. Completed results stay on disk; this launcher intentionally does not automatically resume or restart jobs.


In [ ]:
# (RESULTS/'STOP').touch()


## 7. Export from the SELECTED campaign
The guard prevents the earlier empty-ZIP problem. A complete export requires every requested job to finish. Set `ALLOW_PARTIAL=True` only for deliberate partial inspection.

Includes reports, all per-check certificate tables, calibration tables, anisotropy scalar measurements, model pins/configuration and bounded log tails. Excludes activations/model weights. This is a review archive, not a full resume backup.


In [ ]:
from IPython.display import display,FileLink
ALLOW_PARTIAL=False
ROOT=RESULTS  # Preserve the exact campaign selected above.
plan=json.loads((ROOT/'plan.json').read_text())
exit_info=json.loads((ROOT/'exit.json').read_text()) if (ROOT/'exit.json').exists() else None
complete=bool(exit_info and not exit_info.get('failed',True) and not exit_info.get('stopped',True)
              and len(exit_info.get('jobs',[]))==len(plan['jobs'])
              and all((ROOT/j['name']/'completion.json').exists() for j in plan['jobs']))
if not complete and not ALLOW_PARTIAL:raise RuntimeError('Queue is not fully complete. Use monitor, or explicitly enable a partial export.')
files=set()
for pattern in ['*/reports/*.csv','*/reports/*.pdf','*/reports/*.json','00_recalibration/*.csv','00_recalibration/*.json',
                '*/configuration.json','*/protocol.json','*/environment.json','*/completion.json','*/exit.json',
                '*/*/model.json','*/*/architecture.json','*/*/reference.json']:
    files.update(p for p in ROOT.glob(pattern) if p.is_file())
for name in ['plan.json','started.json','progress.json','exit.json','error.json']:
    if (ROOT/name).exists():files.add(ROOT/name)
reports=[p for p in files if p.suffix=='.csv']
if not reports:raise RuntimeError('No result tables found. Export cancelled.')
status='COMPLETE' if complete else 'PARTIAL'
archive=Path.cwd()/(ROOT.name+'_'+status+'_summaries_'+datetime.now().strftime('%Y%m%d_%H%M%S')+'.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in sorted(files):z.write(p,p.relative_to(ROOT))
    z.write(EVIDENCE/'data/cohort.json','data/cohort.json')
    z.writestr('EXPORT_MANIFEST.json',json.dumps(dict(source=str(ROOT),complete=complete,bundle_sha256=BUNDLE_SHA256,table_count=len(reports)),indent=2))
    for p in [ROOT/'queue.log',ROOT/'cpu.log',*[ROOT/j['name']/'execution.log' for j in plan['jobs']]]:
        if p.exists():
            with p.open('rb') as f:
                f.seek(max(0,p.stat().st_size-20000));z.writestr('log_tails/'+str(p.relative_to(ROOT)),f.read())
with zipfile.ZipFile(archive) as z:assert z.testzip() is None
print('Verified:',status,'| tables:',len(reports),'| MiB:',round(archive.stat().st_size/2**20,2))
display(FileLink(archive.name))


## Optional: CPU calibration only / tests
The bundled evidence is enough for recalibration. These commands do not load model weights. Run the CPU-only command separately if you want its output before launching the GPU queue; use a distinct output directory.


In [ ]:
# CPU-only calibration:
# subprocess.run([str(PYTHON),'-m','precision_study.recalibrate','--evidence',str(EVIDENCE),
#     '--out',str(Path.home()/'threeways_calibration_only')],cwd=CODE,check=True)

# Tiny-model numerical/workflow tests (CPU, no downloads):
# subprocess.run([str(PYTHON),'-m','unittest','discover','-s','tests','-v'],cwd=CODE,check=True)
